# Day 3, Lesson 3.2: A real tool: live Mumbai weather

## Step 1: What an API call looks like

In [1]:
import json
import sys

import requests

sys.path.insert(0, "../src")

response = requests.get(
    "https://api.open-meteo.com/v1/forecast",
    params={
        "latitude": 18.9220, "longitude": 72.8347, "timezone": "Asia/Kolkata",
        "current": "temperature_2m,apparent_temperature,relative_humidity_2m,weather_code",
    },
    timeout=10,
)
print("Full URL:", response.url)
print("HTTP status:", response.status_code, "(200 = OK)")
raw = response.json()
print("\nTop-level keys:", list(raw))
print("current:", raw["current"])
print("units:  ", raw["current_units"])

Full URL: https://api.open-meteo.com/v1/forecast?latitude=18.922&longitude=72.8347&timezone=Asia%2FKolkata&current=temperature_2m%2Capparent_temperature%2Crelative_humidity_2m%2Cweather_code
HTTP status: 200 (200 = OK)

Top-level keys: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'current_units', 'current']
current: {'time': '2026-10-09T15:00', 'interval': 900, 'temperature_2m': 35.8, 'apparent_temperature': 38.4, 'relative_humidity_2m': 36, 'weather_code': 1}
units:   {'time': 'iso8601', 'interval': 'seconds', 'temperature_2m': '°C', 'apparent_temperature': '°C', 'relative_humidity_2m': '%', 'weather_code': 'wmo code'}


## Step 2: Reading the reply

In [2]:
print("We asked for:       18.922, 72.8347")
print("The API used point: ", raw["latitude"], raw["longitude"], "| elevation", raw["elevation"], "m")

We asked for:       18.922, 72.8347
The API used point:  19.015816 72.86982 | elevation 6.0 m


## Step 3: Our tool: `get_live_weather()`

In [3]:
import inspect
import time

from tools import live_weather
from tools.live_weather import ToolError, get_live_weather, recent_hours

start = time.time()
weather = get_live_weather()
first = time.time() - start

start = time.time()
get_live_weather()
second = time.time() - start

print(json.dumps(weather, indent=2, ensure_ascii=False))
print(f"\nFirst call: {first:.2f}s (from the API) | second call: {second:.4f}s (from the cache)")

{
  "location": "Mumbai (South Mumbai)",
  "observed_at": "2026-10-09 15:00 IST",
  "now": {
    "temperature_c": 35.8,
    "feels_like_c": 38.4,
    "humidity_pct": 36,
    "rain_last_15min_mm": 0.0,
    "cloud_cover_pct": 35,
    "wind_kmh": 3.5,
    "condition": "mainly clear"
  },
  "next_hours": [
    {
      "time": "16:00",
      "temperature_c": 34.2,
      "chance_of_rain_pct": 17
    },
    {
      "time": "17:00",
      "temperature_c": 33.4,
      "chance_of_rain_pct": 10
    },
    {
      "time": "18:00",
      "temperature_c": 32.8,
      "chance_of_rain_pct": 6
    },
    {
      "time": "19:00",
      "temperature_c": 32.5,
      "chance_of_rain_pct": 2
    },
    {
      "time": "20:00",
      "temperature_c": 31.7,
      "chance_of_rain_pct": 0
    },
    {
      "time": "21:00",
      "temperature_c": 31.1,
      "chance_of_rain_pct": 0
    }
  ],
  "today": {
    "max_c": 36.0,
    "min_c": 28.3
  },
  "source": "Open-Meteo forecast API (live model data)"
}

First 

In [4]:
recent = recent_hours(24)
print(recent.shape, "|", recent.index.min(), "->", recent.index.max())
recent.tail()

(24, 3) | 2026-10-08 16:00:00 -> 2026-10-09 15:00:00


,temperature_2m,shortwave_radiation,precipitation
time,,,
2026-10-09 11:00:00,35.7,782.0,0.0
2026-10-09 12:00:00,35.8,853.0,0.0
2026-10-09 13:00:00,35.5,709.0,0.0
2026-10-09 14:00:00,36.0,688.0,0.0
2026-10-09 15:00:00,35.4,521.0,0.0


## Step 4: When things go wrong

In [5]:
real_url = live_weather.API_URL
live_weather.API_URL = "https://this-host-does-not-exist.invalid/v1/forecast"
live_weather._cache["data"] = None
try:
    get_live_weather()
except ToolError as error:
    print("ToolError:", error)
finally:
    live_weather.API_URL = real_url
print("Restored. Works again:", get_live_weather()["now"]["temperature_c"], "°C")

Weather API not reachable (attempt 1/3): HTTPSConnectionPool(host='this-host-does-not-exist.invalid', port=443): Max retries exceeded with url: /v1/forecast?latitude=18.922&longitude=72.8347&timezone=Asia%2FKolkata&current=temperature_2m%2Capparent_temperature%2Crelative_humidity_2m%2Cprecipitation%2Cweather_code%2Ccloud_cover%2Cwind_speed_10m&hourly=temperature_2m%2Cshortwave_radiation%2Cprecipitation%2Cprecipitation_probability&daily=temperature_2m_max%2Ctemperature_2m_min&past_days=1&forecast_days=2 (Caused by NameResolutionError("HTTPSConnection(host='this-host-does-not-exist.invalid', port=443): Failed to resolve 'this-host-does-not-exist.invalid' ([Errno 11001] getaddrinfo failed)"))


Weather API not reachable (attempt 2/3): HTTPSConnectionPool(host='this-host-does-not-exist.invalid', port=443): Max retries exceeded with url: /v1/forecast?latitude=18.922&longitude=72.8347&timezone=Asia%2FKolkata&current=temperature_2m%2Capparent_temperature%2Crelative_humidity_2m%2Cprecipitation%2Cweather_code%2Ccloud_cover%2Cwind_speed_10m&hourly=temperature_2m%2Cshortwave_radiation%2Cprecipitation%2Cprecipitation_probability&daily=temperature_2m_max%2Ctemperature_2m_min&past_days=1&forecast_days=2 (Caused by NameResolutionError("HTTPSConnection(host='this-host-does-not-exist.invalid', port=443): Failed to resolve 'this-host-does-not-exist.invalid' ([Errno 11001] getaddrinfo failed)"))


Weather API not reachable (attempt 3/3): HTTPSConnectionPool(host='this-host-does-not-exist.invalid', port=443): Max retries exceeded with url: /v1/forecast?latitude=18.922&longitude=72.8347&timezone=Asia%2FKolkata&current=temperature_2m%2Capparent_temperature%2Crelative_humidity_2m%2Cprecipitation%2Cweather_code%2Ccloud_cover%2Cwind_speed_10m&hourly=temperature_2m%2Cshortwave_radiation%2Cprecipitation%2Cprecipitation_probability&daily=temperature_2m_max%2Ctemperature_2m_min&past_days=1&forecast_days=2 (Caused by NameResolutionError("HTTPSConnection(host='this-host-does-not-exist.invalid', port=443): Failed to resolve 'this-host-does-not-exist.invalid' ([Errno 11001] getaddrinfo failed)"))


ToolError: The live weather service is not reachable right now.


Restored. Works again: 35.8 °C


## Step 5: Gemini + the real tool

In [6]:
import os

from dotenv import load_dotenv
from google import genai
from google.genai import errors, types

load_dotenv("../.env")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
MODEL = "gemini-3.5-flash-lite"

AGENT_RULES = """You are a weather assistant for operations staff in Mumbai.
- You only have live data for Mumbai. For other cities, say you only cover Mumbai.
- Use a tool only when it is needed. Mention the observation time when you give live data.
- If a tool returns an error, tell the user the data is unavailable. Never invent numbers."""

LIVE_WEATHER_TOOL = types.FunctionDeclaration(
    name="get_live_weather",
    description="Get the LIVE weather in Mumbai right now: temperature, feels-like, humidity, rain, "
                "clouds, wind, condition, the next 6 hours (temperature, chance of rain) and today's max/min.",
    parameters_json_schema={"type": "object", "properties": {}},
)
CONFIG = types.GenerateContentConfig(
    system_instruction=AGENT_RULES,
    tools=[types.Tool(function_declarations=[LIVE_WEATHER_TOOL])],
    automatic_function_calling=types.AutomaticFunctionCallingConfig(disable=True),
    temperature=0.1,
)

def run_tool(name):
    try:
        return {"get_live_weather": get_live_weather}[name]()
    except ToolError as error:
        return {"error": str(error)}

def generate(contents, retries=4):
    for attempt in range(1, retries + 1):
        try:
            return client.models.generate_content(model=MODEL, contents=contents, config=CONFIG)
        except errors.APIError as error:
            if error.code in (429, 503) and attempt < retries:
                wait = 30 if error.code == 429 else 5 * attempt
                print(f"   (Gemini {error.code}: waiting {wait}s)")
                time.sleep(wait)
                continue
            raise

def ask_agent(question, max_steps=4):
    contents = [types.Content(role="user", parts=[types.Part.from_text(text=question)])]
    tools_used = []
    for _ in range(max_steps):
        response = generate(contents)
        if not response.function_calls:
            return response.text.strip(), tools_used
        contents.append(response.candidates[0].content)
        parts = []
        for call in response.function_calls:
            tools_used.append(call.name)
            parts.append(types.Part.from_function_response(name=call.name, response=run_tool(call.name)))
        contents.append(types.Content(role="user", parts=parts))
    return "Sorry, I could not finish this question.", tools_used

for question in [
    "How hot is it in Mumbai right now?",
    "Will it rain in Mumbai this evening?",
    "What's the weather like in Delhi?",
]:
    answer, used = ask_agent(question)
    print(f"Q: {question}\n   tools used: {used or 'none'}\n   answer: {answer}\n")

Q: How hot is it in Mumbai right now?
   tools used: ['get_live_weather']
   answer: Right now in Mumbai (observed at 15:00 IST), the temperature is **35.8°C**, but due to the humidity, it feels even hotter at **38.4°C**.



Q: Will it rain in Mumbai this evening?
   tools used: ['get_live_weather']
   answer: Based on the live weather data for Mumbai observed at 15:00 IST on October 9, 2026, the chance of rain this evening is very low. 

Here is the forecast for the upcoming hours:
* **16:00:** 17% chance of rain
* **17:00:** 10% chance of rain
* **18:00:** 6% chance of rain
* **19:00:** 2% chance of rain
* **20:00 – 21:00:** 0% chance of rain

It is expected to remain mainly clear and dry throughout the evening.



Q: What's the weather like in Delhi?
   tools used: none
   answer: I only cover Mumbai, so I don't have weather data for Delhi.

